<!-- notebook-header -->
# Monitoramento e Data Drift

**Modulo:** 06 - MLOps  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Data drift, concept drift, PSI, KS test, thresholds, alertas e retraining.


# 6.3 Monitoramento e Data Drift

Detectando degradacao e mudancas em modelos em producao.

## Prerequisitos - O que Voce Precisa Saber

| Conceito | Por que Importa | Referencia |
|----------|-----------------|----------|
| Distribuicao de probabilidade | Entender mudancas nos dados | 0_6_probabilidade_fundamentos.ipynb |
| Validacao estatistica | Testes para detectar mudancas | 1_2_estatistica_inferencial.ipynb |
| Metricas de avaliacao | Medir degradacao | 3_1_classificacao_completa.ipynb |
| Deploy basico | Modelos ja estao em producao | 6_1_deploy_modelos.ipynb |

## Mapa de Aprendizado

1. Por que modelos degradam? (causa raiz)
2. Data Drift vs Concept Drift (dois tipos diferentes)
3. Model Decay (perda de performance)
4. Deteccao de drift (metricas estatisticas)
5. PSI e KL Divergence (formulas)
6. Detectores de drift (implementacao)
7. Alertas e thresholds (quando reagir)
8. Estrategias de retraining (como se recuperar)
9. Logging estruturado (rastreamento)
10. Dashboards (observabilidade)


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import json

print("=== Monitoramento e Data Drift ===")
print("Detectando degradacao em modelos em producao")
np.random.seed(42)

=== Monitoramento e Data Drift ===
Detectando degradacao em modelos em producao


## 1. Por que Modelos Degradam?

Modelos em producao nao permanecem perfeitos para sempre.

### A Analogia do Termometro

Imagine um termometro que foi calibrado para cidades entre 10C e 30C. Funciona perfeitamente nesse range.

Mas agora voce coloca o termometro no deserto, onde faz 50C. Ou na Antartica, onde faz -40C. Fora do range de calibracao, o termometro para de funcionar confiavel.

Assim funciona com modelos ML: foram calibrados com dados de uma certa distribuicao (treino). Quando a distribuicao muda (producao), a performance degrada.

### Tipos de Degradacao

Existem varias causas diferentes, que precisam de solucoes diferentes.

1. **Data Drift (Covariate Shift)**
   - Distribuicao das features X muda
   - Exemplo: Modelo treinado em clima de 20C, agora vendo 35C

2. **Concept Drift**
   - Relacao entre X e y muda
   - Exemplo: Padrao de fraude muda, velhas regras nao funcionam

3. **Model Decay**
   - Performance cai ao longo do tempo
   - Sem dados novos para retreinar

4. **Data Quality Issues**
   - Valores ausentes aumentam
   - Outliers nao esperados
   - Valores fora do dominio do treino

### O que observar

Na pratica, degradacao aparece como:
- Metrica de performance (accuracy, F1) diminuindo
- Distribuicao de features mudando
- Predicoes ficando inconsistentes
- Taxa de erros aumentando

### O que concluir

Monitoramento e essencial porque:
1. Nao podemos assumir que dados de producao serao identicos aos de treino
2. Mundo real muda constantemente (economia, comportamento, tecnologia)
3. Degradacao silenciosa e perigosa (modelo continua rodando mas produzindo lixo)

### Conexao com

Este conceito conecta com Deploy (6.1) porque um modelo deployado DEVE ser monitorado continuamente.

### Por que em ML

Em ML, drift e degradacao sao problemas fundamentais porque modelos sao treinados uma unica vez em dados historicos. Quando o mundo muda, o modelo nao muda automaticamente.


In [2]:
print("\n=== Data Drift: Mudanca na Distribuicao ===\n")

# Dados de treino (baseline)
np.random.seed(42)
X_train = np.random.normal(loc=0.0, scale=1.0, size=1000)
y_train = (X_train > 0).astype(int)

print("FASE 1: Treino")
print(f"  X_train media: {X_train.mean():.3f}")
print(f"  X_train desvio: {X_train.std():.3f}")
print(f"  X_train min-max: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"  Accuracy no treino: 100%")

# Producao: distribuicao mudou
X_prod_sem_drift = np.random.normal(loc=0.0, scale=1.0, size=500)
X_prod_com_drift = np.random.normal(loc=0.8, scale=1.3, size=500)

print("\nFASE 2: Producao SEM DRIFT")
print(f"  X_prod media: {X_prod_sem_drift.mean():.3f}")
print(f"  X_prod desvio: {X_prod_sem_drift.std():.3f}")
print(f"  X_prod min-max: [{X_prod_sem_drift.min():.3f}, {X_prod_sem_drift.max():.3f}]")

print("\nFASE 3: Producao COM DRIFT")
print(f"  X_prod media: {X_prod_com_drift.mean():.3f}")
print(f"  X_prod desvio: {X_prod_com_drift.std():.3f}")
print(f"  X_prod min-max: [{X_prod_com_drift.min():.3f}, {X_prod_com_drift.max():.3f}]")

print("\nO que observar:")
print("  - Sem drift: media ~0, desvio ~1 (identico ao treino)")
print("  - Com drift: media ~0.8, desvio ~1.3 (diferente!)")

print("\nO que concluir:")
print("  Data Drift = distribuicao de X mudou entre treino e producao")
print("  Comparando estatisticas simples (media, desvio) detectamos isso")

print("\nConexao com:")
print("  Probabilidade (cap 1): distribuicoes definem dados")

print("\nPor que em ML:")
print("  Modelo aprende padrao em treino. Se producao tem distribuicao diferente,")
print("  modelo pode fazer predicoes em regiao nunca vista no treino")


=== Data Drift: Mudanca na Distribuicao ===

FASE 1: Treino
  X_train media: 0.019
  X_train desvio: 0.979
  X_train min-max: [-3.241, 3.853]
  Accuracy no treino: 100%

FASE 2: Producao SEM DRIFT
  X_prod media: 0.108
  X_prod desvio: 1.009
  X_prod min-max: [-2.896, 2.602]

FASE 3: Producao COM DRIFT
  X_prod media: 0.843
  X_prod desvio: 1.278
  X_prod min-max: [-3.023, 4.951]

O que observar:
  - Sem drift: media ~0, desvio ~1 (identico ao treino)
  - Com drift: media ~0.8, desvio ~1.3 (diferente!)

O que concluir:
  Data Drift = distribuicao de X mudou entre treino e producao
  Comparando estatisticas simples (media, desvio) detectamos isso

Conexao com:
  Probabilidade (cap 1): distribuicoes definem dados

Por que em ML:
  Modelo aprende padrao em treino. Se producao tem distribuicao diferente,
  modelo pode fazer predicoes em regiao nunca vista no treino


In [3]:
print("\n=== Teste KS Simulado (sem scipy) ===\n")

def ks_statistic(X1, X2):
    X1_sorted = np.sort(X1)
    X2_sorted = np.sort(X2)
    cdf1 = np.arange(1, len(X1_sorted) + 1) / len(X1_sorted)
    cdf2 = np.arange(1, len(X2_sorted) + 1) / len(X2_sorted)
    all_values = np.sort(np.concatenate([X1, X2]))
    ks_stat = 0.0
    for val in all_values:
        cdf1_val = np.mean(X1 <= val)
        cdf2_val = np.mean(X2 <= val)
        diff = abs(cdf1_val - cdf2_val)
        ks_stat = max(ks_stat, diff)
    return ks_stat

ks_sem_drift = ks_statistic(X_train, X_prod_sem_drift)
ks_com_drift = ks_statistic(X_train, X_prod_com_drift)

print(f"KS Statistic (sem drift): {ks_sem_drift:.4f}")
print(f"KS Statistic (com drift): {ks_com_drift:.4f}")

print("\nInterpretacao:")
print("  KS < 0.1: Sem drift detectado")
print("  KS 0.1-0.2: Possivel drift")
print("  KS > 0.2: Drift confirmado")

print("\nO que observar:")
print(f"  Sem drift: KS = {ks_sem_drift:.4f} (pequeno)")
print(f"  Com drift: KS = {ks_com_drift:.4f} (grande)")

print("\nO que concluir:")
print("  KS test compara distribuicoes de forma nao-parametrica")
print("  E robusto para deteccao de data drift")

print("\nConexao com:")
print("  Testes de hipotese (cap 2): KS e um teste estatistico")

print("\nPor que em ML:")
print("  Podemos usar KS para monitoramento continuo sem assumir distribuicao")


=== Teste KS Simulado (sem scipy) ===

KS Statistic (sem drift): 0.0620
KS Statistic (com drift): 0.3180

Interpretacao:
  KS < 0.1: Sem drift detectado
  KS 0.1-0.2: Possivel drift
  KS > 0.2: Drift confirmado

O que observar:
  Sem drift: KS = 0.0620 (pequeno)
  Com drift: KS = 0.3180 (grande)

O que concluir:
  KS test compara distribuicoes de forma nao-parametrica
  E robusto para deteccao de data drift

Conexao com:
  Testes de hipotese (cap 2): KS e um teste estatistico

Por que em ML:
  Podemos usar KS para monitoramento continuo sem assumir distribuicao


In [4]:
print("\n=== PSI: Population Stability Index ===\n")

def calculate_psi_simple(baseline, current, n_buckets=10):
    min_val = min(baseline.min(), current.min())
    max_val = max(baseline.max(), current.max())
    edges = np.linspace(min_val, max_val, n_buckets + 1)
    baseline_hist, _ = np.histogram(baseline, bins=edges)
    current_hist, _ = np.histogram(current, bins=edges)
    baseline_pct = baseline_hist / baseline_hist.sum()
    current_pct = current_hist / current_hist.sum()
    baseline_pct = np.where(baseline_pct == 0, 1e-5, baseline_pct)
    current_pct = np.where(current_pct == 0, 1e-5, current_pct)
    psi = np.sum((current_pct - baseline_pct) * np.log(current_pct / baseline_pct))
    return psi

psi_sem_drift = calculate_psi_simple(X_train, X_prod_sem_drift)
psi_com_drift = calculate_psi_simple(X_train, X_prod_com_drift)

print(f"PSI (sem drift): {psi_sem_drift:.4f}")
print(f"PSI (com drift): {psi_com_drift:.4f}")

print("\nInterpretacao:")
print("  PSI < 0.1: Sem drift")
print("  PSI 0.1-0.25: Possivel drift (monitorar)")
print("  PSI > 0.25: Drift confirmado (agir!)")

print("\nO que observar:")
print(f"  Sem drift: PSI = {psi_sem_drift:.4f} (green zone)")
print(f"  Com drift: PSI = {psi_com_drift:.4f} (red zone)")

print("\nO que concluir:")
print("  PSI e mais sensivel que KS para quantificar mudanca")
print("  E util como metrica continua de monitoramento")

print("\nConexao com:")
print("  KL Divergence (teoria da informacao): PSI e variacao disso")

print("\nPor que em ML:")
print("  PSI fornece numero unico para 'saude' da distribuicao")


=== PSI: Population Stability Index ===

PSI (sem drift): 0.0294
PSI (com drift): 0.6668

Interpretacao:
  PSI < 0.1: Sem drift
  PSI 0.1-0.25: Possivel drift (monitorar)
  PSI > 0.25: Drift confirmado (agir!)

O que observar:
  Sem drift: PSI = 0.0294 (green zone)
  Com drift: PSI = 0.6668 (red zone)

O que concluir:
  PSI e mais sensivel que KS para quantificar mudanca
  E util como metrica continua de monitoramento

Conexao com:
  KL Divergence (teoria da informacao): PSI e variacao disso

Por que em ML:
  PSI fornece numero unico para 'saude' da distribuicao


## 2. Concept Drift - Mudanca na Relacao X -> y

Quando a relacao entre features e alvo muda.

### Tipos de Concept Drift

1. **Gradual Drift**
   - Mudanca lenta e continua ao longo do tempo
   - Exemplo: Preferencias de clientes evoluem gradualmente

2. **Abrupto (Sudden)**
   - Mudanca subita em um momento especifico
   - Exemplo: Novo competidor entra no mercado

3. **Recorrente (Seasonal)**
   - Padrao muda periodicamente
   - Exemplo: Vendas variam por estacao

4. **Incremental**
   - Pequenas mudancas acumuladas
   - Exemplo: Tecnologia muda lentamente

### O que observar

Concept drift se manifesta como:
- Accuracy cai mas distribuicao de X parece estavel
- Mesmos valores de X dao predicoes erradas agora
- False positives ou false negatives aumentam

### O que concluir

Concept drift e mais perigoso que data drift porque:
- Data drift: mudar distribuicao, mas relacao permanece. Retreinar ajuda.
- Concept drift: relacao mudou. Retreinar sozinho pode nao resolver.

Exemplos:
- Fraude: novos tipos de fraude surgem, velhos padroes nao funcionam
- Economia: criterios de risco crediticio mudam com crise economica
- Saude: criterios diagnosticos evoluem com pesquisa

### Conexao com

Validacao cruzada (cap 3): Dividir dados no tempo (nao aleatoriamente) ajuda detectar concept drift.

### Por que em ML

Mundo real tem dinamica. Relacoes causais mudam quando sistemas mudam fundamentalmente.


In [5]:
print("\n=== Concept Drift Simulado ===\n")

np.random.seed(42)
X_train = np.random.normal(0, 1, 100)
y_train = (X_train > 0).astype(int)

train_acc = np.mean(y_train == (X_train > 0).astype(int))
print(f"TREINO: Relacao y = 1 se X > 0")
print(f"  Accuracy: {train_acc:.1%}")
print(f"  X_train media: {X_train.mean():.3f}")

X_prod = np.random.normal(0, 1, 100)
y_prod_true = (X_prod < 0).astype(int)
y_pred_old = (X_prod > 0).astype(int)
prod_acc_old = np.mean(y_prod_true == y_pred_old)

print(f"\nPRODUCAO: Relacao mudou para y = 1 se X < 0")
print(f"  X_prod media: {X_prod.mean():.3f}")
print(f"  Accuracy com modelo antigo: {prod_acc_old:.1%} (DESASTRE!)")

y_pred_new = (X_prod < 0).astype(int)
prod_acc_new = np.mean(y_prod_true == y_pred_new)
print(f"  Accuracy com modelo novo: {prod_acc_new:.1%} (resolvido)")

print("\nO que observar:")
print(f"  Degradacao de {train_acc:.0%} para {prod_acc_old:.0%}")
print(f"  Distribuicao de X nao mudou (media ~0)")
print(f"  Mas accuracy caiu para {prod_acc_old:.0%}")

print("\nO que concluir:")
print("  Isso e Concept Drift: relacao mudou, nao dados")
print("  PSI/KS nao detectariam (distribuicao e estavel)")
print("  Precisa monitorar PERFORMANCE, nao apenas distribuicao")

print("\nConexao com:")
print("  Avaliacao de modelos (cap 3): metrics sao primeiras pistas")

print("\nPor que em ML:")
print("  Concept drift e invisivel em dados. Precisa monitorar saida do modelo")


=== Concept Drift Simulado ===

TREINO: Relacao y = 1 se X > 0
  Accuracy: 100.0%
  X_train media: -0.104

PRODUCAO: Relacao mudou para y = 1 se X < 0
  X_prod media: 0.022
  Accuracy com modelo antigo: 0.0% (DESASTRE!)
  Accuracy com modelo novo: 100.0% (resolvido)

O que observar:
  Degradacao de 100% para 0%
  Distribuicao de X nao mudou (media ~0)
  Mas accuracy caiu para 0%

O que concluir:
  Isso e Concept Drift: relacao mudou, nao dados
  PSI/KS nao detectariam (distribuicao e estavel)
  Precisa monitorar PERFORMANCE, nao apenas distribuicao

Conexao com:
  Avaliacao de modelos (cap 3): metrics sao primeiras pistas

Por que em ML:
  Concept drift e invisivel em dados. Precisa monitorar saida do modelo


## 3. Model Decay - Degradacao de Performance

Acompanhar performance ao longo do tempo.

### O que observar

Model decay aparece como queda continua em metricas como:
- Accuracy
- Precision
- Recall
- F1-score

### O que concluir

Degradacao constante e sinal de que dados estao mudando (data ou concept drift).

### Conexao com

Validacao (cap 3): precisamos de metricas confiáveis para detectar decay.

### Por que em ML

Se nao monitorar continuamente, modelo degradado continuara sendo usado atendendo usuarios com predicoes ruins.


In [6]:
print("\n=== Model Decay: Degradacao de Performance ===\n")

dias = np.arange(0, 90, 5)
performance_base = 0.95
performance = performance_base - 0.0003 * dias + np.random.normal(0, 0.005, len(dias))
performance = np.clip(performance, 0, 1)

print("Performance ao longo de 3 meses:")
for dia, perf in zip(dias, performance):
    stars = int(perf * 10)
    print(f"  Dia {dia:3d}: {perf:.3f} [{stars}{'.'*(10-stars)}]")

degradacao_total = performance_base - performance[-1]
taxa_degradacao = degradacao_total / len(dias)

print(f"\nDegradacao total: {degradacao_total:.3f} ({degradacao_total*100:.1f}%)")
print(f"Taxa media: {taxa_degradacao:.4f} por dia")

limiar = 0.90
dias_acima_limiar = dias[performance >= limiar]
if len(dias_acima_limiar) < len(dias):
    primeiro_dia_abaixo = dias[performance < limiar][0]
    print(f"\nPerformance caiu abaixo de {limiar:.0%} no dia {primeiro_dia_abaixo}")
else:
    print(f"\nPerformance manteve-se acima de {limiar:.0%} todo periodo")

print("\nO que observar:")
print("  - Performance nao e constante (degrada)")
print("  - Queda e aproximadamente linear")
print("  - Ruido natural em volta da tendencia")

print("\nO que concluir:")
print("  Modelo precisa ser retreinado antes de passar limiar critico")
print("  Monitoramento precisa ser continuo, nao apenas periodico")

print("\nConexao com:")
print("  Metricas e thresholds (cap 5): definir limiar e essencial")

print("\nPor que em ML:")
print("  Esperar modelo falhar completamente antes de agir e arriscado")


=== Model Decay: Degradacao de Performance ===

Performance ao longo de 3 meses:
  Dia   0: 0.952 [9.]
  Dia   5: 0.951 [9.]
  Dia  10: 0.952 [9.]
  Dia  15: 0.951 [9.]
  Dia  20: 0.937 [9.]
  Dia  25: 0.938 [9.]
  Dia  30: 0.944 [9.]
  Dia  35: 0.942 [9.]
  Dia  40: 0.941 [9.]
  Dia  45: 0.956 [9.]
  Dia  50: 0.938 [9.]
  Dia  55: 0.939 [9.]
  Dia  60: 0.937 [9.]
  Dia  65: 0.934 [9.]
  Dia  70: 0.927 [9.]
  Dia  75: 0.931 [9.]
  Dia  80: 0.922 [9.]
  Dia  85: 0.923 [9.]

Degradacao total: 0.027 (2.7%)
Taxa media: 0.0015 por dia

Performance manteve-se acima de 90% todo periodo

O que observar:
  - Performance nao e constante (degrada)
  - Queda e aproximadamente linear
  - Ruido natural em volta da tendencia

O que concluir:
  Modelo precisa ser retreinado antes de passar limiar critico
  Monitoramento precisa ser continuo, nao apenas periodico

Conexao com:
  Metricas e thresholds (cap 5): definir limiar e essencial

Por que em ML:
  Esperar modelo falhar completamente antes de agi

In [7]:
print("\n=== Feature Drift: Monitorar Cada Feature ===\n")

n_features = 3
n_train = 500
n_prod = 300

X_train_multi = np.random.normal(loc=0, scale=1, size=(n_train, n_features))

# Create production data - correctly sized
X_prod_multi = np.random.normal(loc=0, scale=1, size=(n_prod, n_features))
# Override feature 1 to have drift
X_prod_multi[:, 1] = np.random.normal(loc=1.5, scale=0.8, size=n_prod)

print("Feature-wise Drift Detection:")
print("-" * 50)

for feat_idx in range(n_features):
    ks_feat = ks_statistic(X_train_multi[:, feat_idx], X_prod_multi[:, feat_idx])
    psi_feat = calculate_psi_simple(X_train_multi[:, feat_idx],
                                     X_prod_multi[:, feat_idx])

    drift_status = "DRIFT!" if ks_feat > 0.15 else "OK"

    print(f"\nFeature {feat_idx}:")
    print(f"  KS Statistic: {ks_feat:.4f}")
    print(f"  PSI: {psi_feat:.4f}")
    print(f"  Status: {drift_status}")

print("\nO que observar:")
print("  Feature 1 e 0 aparecem estavel")
print("  Feature 1 tem drift significativo")

print("\nO que concluir:")
print("  Drift pode afetar algumas features e nao outras")
print("  Investigacao: o que mudou exatamente no Feature 1?")
print("  Pode revelar mudanca real no mundo (ex: nova fonte de dados)")

print("\nConexao com:")
print("  Feature engineering (cap 4): entender cada feature")

print("\nPor que em ML:")
print("  Drift granular ajuda diagnosticar causa raiz da degradacao")


=== Feature Drift: Monitorar Cada Feature ===

Feature-wise Drift Detection:
--------------------------------------------------

Feature 0:
  KS Statistic: 0.0520
  PSI: 0.0935
  Status: OK

Feature 1:
  KS Statistic: 0.6087
  PSI: 2.5937
  Status: DRIFT!

Feature 2:
  KS Statistic: 0.0753
  PSI: 0.0328
  Status: OK

O que observar:
  Feature 1 e 0 aparecem estavel
  Feature 1 tem drift significativo

O que concluir:
  Drift pode afetar algumas features e nao outras
  Investigacao: o que mudou exatamente no Feature 1?
  Pode revelar mudanca real no mundo (ex: nova fonte de dados)

Conexao com:
  Feature engineering (cap 4): entender cada feature

Por que em ML:
  Drift granular ajuda diagnosticar causa raiz da degradacao


## 4. Sistema de Alertas

Detectar drift e disparar alertas automaticos.

### Tipos de Alertas

1. **Green Zone**: Tudo OK, continue monitorando
2. **Yellow Zone**: Possivel drift detectado, investigar
3. **Red Zone**: Drift confirmado, acao imediata necessaria

### O que observar

Alertas precisam balancear:
- **Sensibilidade**: nao perder drifts reais (poucos falsos negativos)
- **Especificidade**: nao falsos alarmes (poucos falsos positivos)

### O que concluir

Thresholds devem ser calibrados para seu dominio e risco.
Drift critico em saude requer threshold baixo. Recomendacao de produto pode tolerar drift maior.

### Conexao com

Tomada de decisao: threshold define quando sistema reage.

### Por que em ML

Alertas automáticos sao essenciais porque humanos nao conseguem monitorar 24/7.


In [8]:
print("\n=== Sistema de Alertas ===\n")

class MonitorDrift:
    def __init__(self, ks_threshold=0.15, psi_threshold=0.25,
                 accuracy_threshold=0.90):
        self.ks_threshold = ks_threshold
        self.psi_threshold = psi_threshold
        self.accuracy_threshold = accuracy_threshold
        self.alerts = []

    def check_drift(self, baseline, current, feature_name=""):
        ks_stat = ks_statistic(baseline, current)
        psi_stat = calculate_psi_simple(baseline, current)
        status = "GREEN"
        if ks_stat > self.ks_threshold or psi_stat > self.psi_threshold:
            status = "RED"
            alert_msg = f"{feature_name} DRIFT: KS={ks_stat:.4f}, PSI={psi_stat:.4f}"
            self.alerts.append(alert_msg)
        elif ks_stat > 0.08 or psi_stat > 0.15:
            status = "YELLOW"
        return status, ks_stat, psi_stat

    def check_performance(self, current_acc):
        if current_acc < self.accuracy_threshold:
            alert_msg = f"PERFORMANCE: Accuracy {current_acc:.3f} abaixo {self.accuracy_threshold}"
            self.alerts.append(alert_msg)
            return "RED"
        return "GREEN"

    def report(self):
        return {
            "n_alerts": len(self.alerts),
            "status": "CRITICAL" if len(self.alerts) > 0 else "HEALTHY",
            "alerts": self.alerts
        }

monitor = MonitorDrift(ks_threshold=0.15, psi_threshold=0.25)

print("Verificacao 1: Sem drift")
status, ks, psi = monitor.check_drift(X_train, X_prod_sem_drift, "Feature_A")
print(f"  Status: {status}, KS={ks:.4f}, PSI={psi:.4f}")

print("\nVerificacao 2: Com drift")
status, ks, psi = monitor.check_drift(X_train, X_prod_com_drift, "Feature_B")
print(f"  Status: {status}, KS={ks:.4f}, PSI={psi:.4f}")

print("\nVerificacao 3: Performance degradada")
status = monitor.check_performance(0.88)
print(f"  Status: {status}")

print("\nRelatorio de Alertas:")
report = monitor.report()
print(f"  Sistema: {report['status']}")
print(f"  Total de alertas: {report['n_alerts']}")
for alert in report['alerts']:
    print(f"    - {alert}")

print("\nO que observar:")
print("  - Alertas sao disparados quando drift e detectado")
print("  - Sistema diferencia GREEN/YELLOW/RED")

print("\nO que concluir:")
print("  Alertas automáticos permitem reacao rápida")
print("  Thresholds precisam ser ajustados por trial-and-error")

print("\nConexao com:")
print("  Monitoramento em producao: integra-se com observabilidade")

print("\nPor que em ML:")
print("  Sistemas de alerta transformam deteccao em acao")


=== Sistema de Alertas ===

Verificacao 1: Sem drift
  Status: RED, KS=0.1540, PSI=0.3928

Verificacao 2: Com drift
  Status: RED, KS=0.4080, PSI=1.4129

Verificacao 3: Performance degradada
  Status: RED

Relatorio de Alertas:
  Sistema: CRITICAL
  Total de alertas: 3
    - Feature_A DRIFT: KS=0.1540, PSI=0.3928
    - Feature_B DRIFT: KS=0.4080, PSI=1.4129
    - PERFORMANCE: Accuracy 0.880 abaixo 0.9

O que observar:
  - Alertas sao disparados quando drift e detectado
  - Sistema diferencia GREEN/YELLOW/RED

O que concluir:
  Alertas automáticos permitem reacao rápida
  Thresholds precisam ser ajustados por trial-and-error

Conexao com:
  Monitoramento em producao: integra-se com observabilidade

Por que em ML:
  Sistemas de alerta transformam deteccao em acao


## 5. Estrategias de Retraining

Como e quando retreinar o modelo.

### Estrategias

1. **Scheduled Retraining**
   - Retreinar em intervalo fixo (ex: diariamente, semanalmente)
   - Vantagem: Simples de implementar
   - Desvantagem: Pode desperdicar recursos ou ser lento demais

2. **Triggered Retraining**
   - Retreinar quando alerta de drift/degradacao dispara
   - Vantagem: Eficiente em recursos, reage as mudancas
   - Desvantagem: Pode ser tarde demais se drift e rápido

3. **Incremental Retraining**
   - Retreinar com dados novos, mantendo conhecimento antigo
   - Vantagem: Balanco entre estabilidade e adaptacao
   - Desvantagem: Mais complexo de implementar

4. **Active Learning**
   - Selecionar amostras mais informativas para labeling
   - Vantagem: Diminui custo de labeling
   - Desvantagem: Muito complexo

### O que observar

Cada estrategia tem trade-offs:
- Frequencia vs. custo
- Precisao vs. estabilidade
- Latencia vs. recursos

### O que concluir

Escolha depende do dominio:
- Saude: Triggered (rápida resposta)
- Recomendacao: Scheduled (menos critico)
- Fraude: Triggered + Incremental (balance)

### Conexao com

Deploy (6.1): Processo de retraining integra-se com deployment.

### Por que em ML

Sem estrategia clara, sistema fica sem direcao quando drift aparece.


In [9]:
print("\n=== Estrategias de Retraining ===\n")

class RetrainingScheduler:
    def __init__(self, strategy="scheduled", interval_days=7):
        self.strategy = strategy
        self.interval_days = interval_days
        self.last_training_day = 0
        self.last_drift_alert_day = None
        self.retraining_history = []

    def should_retrain_scheduled(self, current_day):
        return (current_day - self.last_training_day) >= self.interval_days

    def should_retrain_triggered(self, current_day, drift_detected):
        if drift_detected:
            self.last_drift_alert_day = current_day
            return True
        return False

    def should_retrain(self, current_day, drift_detected=False):
        if self.strategy == "scheduled":
            return self.should_retrain_scheduled(current_day)
        elif self.strategy == "triggered":
            return self.should_retrain_triggered(current_day, drift_detected)
        return False

    def record_training(self, current_day):
        self.last_training_day = current_day
        self.retraining_history.append(current_day)

scheduler_scheduled = RetrainingScheduler(strategy="scheduled", interval_days=7)
scheduler_triggered = RetrainingScheduler(strategy="triggered")

drift_events = [15, 25, 45, 60, 75]

print("Estrategia 1: SCHEDULED (retrain a cada 7 dias)")
print("-" * 50)
for day in range(0, 100, 7):
    scheduler_scheduled.record_training(day)
print(f"Dias com retraining: {scheduler_scheduled.retraining_history}")
print(f"Total: {len(scheduler_scheduled.retraining_history)} vezes")

print("\nEstrategia 2: TRIGGERED (retrain quando drift detectado)")
print("-" * 50)
for day in drift_events:
    scheduler_triggered.record_training(day)
print(f"Dias com retraining: {scheduler_triggered.retraining_history}")
print(f"Total: {len(scheduler_triggered.retraining_history)} vezes")

print("\nComparacao:")
print(f"  Scheduled: {len(scheduler_scheduled.retraining_history)} entrenamientos em 100 dias")
print(f"  Triggered: {len(scheduler_triggered.retraining_history)} entrenamientos em 100 dias")
print(f"  Economia: {len(scheduler_scheduled.retraining_history) - len(scheduler_triggered.retraining_history)} menos com Triggered")

print("\nO que observar:")
print("  - Scheduled e previsivel")
print("  - Triggered reage apenas quando necessario")

print("\nO que concluir:")
print("  Triggered economiza recursos mas precisa de bom detector")
print("  Hybrid: combinar ambas estrategias")

print("\nConexao com:")
print("  Scheduling e DevOps: automatizar retraining")

print("\nPor que em ML:")
print("  Retraining e caro. Precisa de estrategia clara")


=== Estrategias de Retraining ===

Estrategia 1: SCHEDULED (retrain a cada 7 dias)
--------------------------------------------------
Dias com retraining: [0, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98]
Total: 15 vezes

Estrategia 2: TRIGGERED (retrain quando drift detectado)
--------------------------------------------------
Dias com retraining: [15, 25, 45, 60, 75]
Total: 5 vezes

Comparacao:
  Scheduled: 15 entrenamientos em 100 dias
  Triggered: 5 entrenamientos em 100 dias
  Economia: 10 menos com Triggered

O que observar:
  - Scheduled e previsivel
  - Triggered reage apenas quando necessario

O que concluir:
  Triggered economiza recursos mas precisa de bom detector
  Hybrid: combinar ambas estrategias

Conexao com:
  Scheduling e DevOps: automatizar retraining

Por que em ML:
  Retraining e caro. Precisa de estrategia clara


## 6. Logging Estruturado

Rastrear tudo estruturadamente para observabilidade.

### O que Registrar

1. **Predicoes**: request_id, features, output, confianca
2. **Performance**: accuracy, precision, recall, F1 (periodico)
3. **Alertas**: tipo, valor, threshold, timestamp
4. **Retraining**: quando ocorreu, qual dados usou, performance antes/depois

### O que observar

Logs estruturados (JSON) sao essenciais para:
- Debugging rápido
- Analise de causa raiz
- Conformidade regulatoria
- Reproducao de problemas

### O que concluir

Logging nao e luxo, e necessidade em producao.

### Conexao com

Observabilidade: logs sao camada fundamental de observacao.

### Por que em ML

Sem logs, e impossivel entender o que deu errado quando algo falha.


In [10]:
print("\n=== Logging Estruturado ===\n")

class MLLogger:
    def __init__(self):
        self.logs = []

    def log_prediction(self, request_id, features, prediction, confidence):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id,
            "event": "prediction",
            "n_features": len(features),
            "prediction": int(prediction),
            "confidence": float(confidence)
        }
        self.logs.append(log_entry)

    def log_metrics(self, accuracy, precision, recall, n_samples):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "event": "performance",
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "n_samples": int(n_samples)
        }
        self.logs.append(log_entry)

    def log_drift_alert(self, metric_name, value, threshold, feature=""):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "event": "drift_alert",
            "metric": metric_name,
            "feature": feature,
            "value": float(value),
            "threshold": float(threshold),
            "severity": "HIGH"
        }
        self.logs.append(log_entry)

    def log_retraining(self, accuracy_before, accuracy_after, n_samples):
        improvement = accuracy_after - accuracy_before
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "event": "retraining",
            "accuracy_before": float(accuracy_before),
            "accuracy_after": float(accuracy_after),
            "improvement": float(improvement),
            "n_samples": int(n_samples)
        }
        self.logs.append(log_entry)

    def get_logs_as_json(self):
        return json.dumps(self.logs, indent=2)

    def summary(self):
        by_event = {}
        for log in self.logs:
            event = log.get("event")
            by_event[event] = by_event.get(event, 0) + 1
        return by_event

logger = MLLogger()

logger.log_prediction("req_001", [0.5, 1.2, -0.3], 1, 0.87)
logger.log_prediction("req_002", [0.1, 0.8, 0.2], 0, 0.92)
logger.log_metrics(accuracy=0.92, precision=0.89, recall=0.91, n_samples=1000)
logger.log_drift_alert("KS", 0.18, 0.15, "Feature_1")
logger.log_retraining(accuracy_before=0.92, accuracy_after=0.94, n_samples=500)

print("Resumo de eventos:")
summary = logger.summary()
for event_type, count in summary.items():
    print(f"  {event_type}: {count}")

print("\nPrimeiros 3 logs (JSON):")
print(json.dumps(logger.logs[:3], indent=2))

print("\nO que observar:")
print("  - Logs estruturados com timestamps")
print("  - Rastreavel para cada predicao/evento")

print("\nO que concluir:")
print("  Logs sao essenciais para audit trail")
print("  Permitem reconstruir historico completo")

print("\nConexao com:")
print("  Observabilidade em producao")

print("\nPor que em ML:")
print("  Compliance, debugging, auditoria")


=== Logging Estruturado ===

Resumo de eventos:
  prediction: 2
  performance: 1
  drift_alert: 1
  retraining: 1

Primeiros 3 logs (JSON):
[
  {
    "timestamp": "2026-05-18T23:48:21.636537",
    "request_id": "req_001",
    "event": "prediction",
    "n_features": 3,
    "prediction": 1,
    "confidence": 0.87
  },
  {
    "timestamp": "2026-05-18T23:48:21.636558",
    "request_id": "req_002",
    "event": "prediction",
    "n_features": 3,
    "prediction": 0,
    "confidence": 0.92
  },
  {
    "timestamp": "2026-05-18T23:48:21.636574",
    "event": "performance",
    "accuracy": 0.92,
    "precision": 0.89,
    "recall": 0.91,
    "n_samples": 1000
  }
]

O que observar:
  - Logs estruturados com timestamps
  - Rastreavel para cada predicao/evento

O que concluir:
  Logs sao essenciais para audit trail
  Permitem reconstruir historico completo

Conexao com:
  Observabilidade em producao

Por que em ML:
  Compliance, debugging, auditoria


## 7. Dashboard de Monitoramento

Visualizar saude do modelo em tempo real.

### Componentes

1. **KPIs**: Accuracy, F1-score, latencia
2. **Trend Charts**: Performance ao longo do tempo
3. **Drift Indicators**: PSI, KS test por feature
4. **Alerts Table**: Ultimos alertas disparados
5. **Retraining History**: Quando modelo foi retreinado

### O que observar

Dashboard deve ser:
- Simples (maximo 5 metricas principais)
- Atualizado em tempo real
- Alertas visiveis
- Historico de 30-90 dias

### O que concluir

Dashboard viabiliza observabilidade continua.

### Conexao com

DevOps: Dashboard e ferramenta de operacoes.

### Por que em ML

Modelo e componente vivo. Precisa monitoramento como qualquer aplicacao.


In [11]:
print("\n=== Feature Drift: Monitorar Cada Feature ===\n")

n_features = 3
n_train = 500
n_prod = 300

X_train_multi = np.random.normal(loc=0, scale=1, size=(n_train, n_features))

# Create production data: first part same, feature 1 has drift
X_prod_multi = np.random.normal(loc=0, scale=1, size=(n_prod, n_features))
X_prod_multi[:, 1] = np.random.normal(loc=1.5, scale=0.8, size=n_prod)

print("Feature-wise Drift Detection:")
print("-" * 50)

for feat_idx in range(n_features):
    ks_feat = ks_statistic(X_train_multi[:, feat_idx], X_prod_multi[:, feat_idx])
    psi_feat = calculate_psi_simple(X_train_multi[:, feat_idx],
                                     X_prod_multi[:, feat_idx])

    drift_status = "DRIFT!" if ks_feat > 0.15 else "OK"

    print(f"\nFeature {feat_idx}:")
    print(f"  KS Statistic: {ks_feat:.4f}")
    print(f"  PSI: {psi_feat:.4f}")
    print(f"  Status: {drift_status}")

print("\nO que observar:")
print("  Feature 1 e 0 aparecem estavel")
print("  Feature 1 tem drift significativo")

print("\nO que concluir:")
print("  Drift pode afetar algumas features e nao outras")
print("  Investigacao: o que mudou exatamente no Feature 1?")
print("  Pode revelar mudanca real no mundo (ex: nova fonte de dados)")

print("\nConexao com:")
print("  Feature engineering (cap 4): entender cada feature")

print("\nPor que em ML:")
print("  Drift granular ajuda diagnosticar causa raiz da degradacao")


=== Feature Drift: Monitorar Cada Feature ===

Feature-wise Drift Detection:
--------------------------------------------------

Feature 0:
  KS Statistic: 0.0387
  PSI: 0.0488
  Status: OK

Feature 1:
  KS Statistic: 0.6307
  PSI: 4.3124
  Status: DRIFT!

Feature 2:
  KS Statistic: 0.0633
  PSI: 0.0763
  Status: OK

O que observar:
  Feature 1 e 0 aparecem estavel
  Feature 1 tem drift significativo

O que concluir:
  Drift pode afetar algumas features e nao outras
  Investigacao: o que mudou exatamente no Feature 1?
  Pode revelar mudanca real no mundo (ex: nova fonte de dados)

Conexao com:
  Feature engineering (cap 4): entender cada feature

Por que em ML:
  Drift granular ajuda diagnosticar causa raiz da degradacao


## 8. Erros Comuns e Armadilhas


### Erro 1: Nao Rastrear Baseline do Treino

**Problema**: Calcula PSI mas nao tem dados de treino para comparar.

**Causa**: Nao guardou distribuicao/estatisticas dos dados de treino.

**Como evitar**:
```
baseline = {
    "mean": X_train.mean(axis=0),
    "std": X_train.std(axis=0),
    "min": X_train.min(axis=0),
    "max": X_train.max(axis=0)
}
np.save("baseline.npy", baseline)
```

**O que observar**: Sem baseline, nao ha referencia para comparacao.

**O que concluir**: Baseline e tao importante quanto o modelo.

**Conexao com**: Deploy (6.1) - sempre guardar artefatos de treino.

**Por que em ML**: Monitoramento e comparacao. Sem baseline, nao ha baseline.


### Erro 2: Confundir Data Drift com Concept Drift

**Problema**: Detectou data drift, retreinou modelo, accuracy nao melhorou.

**Causa**: Era concept drift, nao data drift! Apenas retreinar nao resolve.

**Como evitar**:
1. Se performance CAIU mas PSI/KS nao mudou muito: concept drift
2. Se performance estavel mas PSI/KS alto: data drift puro (retreinar ajuda)

**Exemplo**:
- Data Drift: X mudou, relacao X->y mesma. Retreinar=solucao
- Concept Drift: X estavel, relacao X->y mudou. Retreinar pode nao resolver

**O que observar**: Correlacao entre drift metrics e performance.

**O que concluir**: Diagnostico correto e essencial para acao correta.

**Conexao com**: Causas raiz (investigacao).

**Por que em ML**: Diferentes causas requerem diferentes solucoes.


### Erro 3: Monitorar Apenas Drift, Nao Performance

**Problema**: PSI monitora, accuracy nao. Detecta data drift mas nao concept drift.

**Causa**: Concept drift e invisivel em distribuicao de X.

**Como evitar**: Sempre monitore AMBAS:
- Data Drift: PSI, KS test nas features
- Performance: Accuracy, F1, precision, recall

**O que observar**: Alertas devem combinar multiplas sinais.

**O que concluir**: Monitoramento multi-sinal e mais robusto.

**Conexao com**: Robustez de monitoramento.

**Por que em ML**: Um sinal nao e suficiente para decisao confiavel.


### Erro 4: Ignorar Sazonalidade e Padroes Naturais

**Problema**: Dispara alerta de drift toda segunda-feira (falso positivo).

**Causa**: Padroes naturais (semanal, sazonal) sao confundidos com drift real.

**Como evitar**:
1. Conhecer dados! Documento padroes sazonais
2. Comparar com mesmo periodo anterior (segunda-feira com segunda-feira anterior)
3. Usar baseline por periodo (nao media global)

**O que observar**: Padroes periodicos em dados.

**O que concluir**: Sazonalidade pode mascarar drift real.

**Conexao com**: Analise exploratoria de dados (cap 1).

**Por que em ML**: Mundo tem ritmo natural. Distinguir ritmo de mudanca.


### Erro 5: Retraining Muito Frequente

**Problema**: Retrain todo dia porque detecta micro-drift. Caro em recursos.

**Causa**: Threshold muito baixo (sensivel demais).

**Como evitar**:
1. Calibrar threshold para seu dominio
2. Usar estrategia triggered (nao scheduled) quando possivel
3. Retraining incremental (nao do zero)

**O que observar**: Custo-beneficio de retraining.

**O que concluir**: Threshold deve equilibrar sensibilidade e custo.

**Conexao com**: Recursos computacionais.

**Por que em ML**: Retraining e caro. Precisa usar com sabedoria.


### Erro 6: Nao Manter Historico para Reproducibilidade

**Problema**: Modelo degradou no mes passado, ninguem consegue reproducir problema.

**Causa**: Nao guardou versoes do modelo, dados usados, metricas.

**Como evitar**: Manter registro completo:
```
training_record = {
    "timestamp": datetime.now(),
    "model_version": "v1.2.3",
    "data_file": "data_2024_03_01.parquet",
    "train_samples": 10000,
    "baseline_accuracy": 0.95
}
```

**O que observar**: Rastreabilidade completa.

**O que concluir**: Sem historico, nao ha como debugar.

**Conexao com**: Versionamento de modelos (MLOps).

**Por que em ML**: Reproducibilidade e essencial para confianca.


## 9. Exercicios Praticos

### TAREFA DO ALUNO Exercicio 1: Detectar Data Drift

Crie dados com distribuicoes diferentes e implemente detector de data drift.

**Tarefa**:
1. Gerar X_train com distribuicao Normal(0, 1)
2. Gerar X_prod com distribuicao Normal(0.5, 1.2)
3. Calcular KS statistic e PSI
4. Disparar alerta se PSI > 0.25 ou KS > 0.15

**Saida esperada**: Alerta de drift detectado


In [12]:
# TAREFA DO ALUNO: Calcular KS statistic entre dois datasets com distribuicoes diferentes
result = None  # Preencher
print(f'Exercicio 1: {result}')

Exercicio 1: None


In [13]:
# TAREFA DO ALUNO: Implementar PSI calculation para 3 features e encontrar qual tem maior drift
result = None  # Preencher
print(f'Exercicio 2: {result}')

Exercicio 2: None


In [14]:
# TAREFA DO ALUNO: Criar MonitorDrift que alerta quando PSI > 0.25 em multiplas features
result = None  # Preencher
print(f'Exercicio 3: {result}')

Exercicio 3: None


In [15]:
# TAREFA DO ALUNO: Calcular KS statistic entre dois datasets com distribuicoes diferentes
result = None  # Preencher
print(f'Exercicio 1: {result}')

Exercicio 1: None


In [16]:
# TAREFA DO ALUNO: Implementar PSI calculation para 3 features e encontrar qual tem maior drift
result = None  # Preencher
print(f'Exercicio 2: {result}')

Exercicio 2: None


In [17]:
# TAREFA DO ALUNO: Criar MonitorDrift que alerta quando PSI > 0.25 em multiplas features
result = None  # Preencher
print(f'Exercicio 3: {result}')

Exercicio 3: None


In [18]:
print("\n=== SOLUCAO: Exercicio 1 - Detector de Data Drift ===\n")

X_train_ex1 = np.random.normal(loc=0.0, scale=1.0, size=500)
X_prod_ex1 = np.random.normal(loc=0.5, scale=1.2, size=500)

ks_ex1 = ks_statistic(X_train_ex1, X_prod_ex1)
psi_ex1 = calculate_psi_simple(X_train_ex1, X_prod_ex1)

print(f"KS Statistic: {ks_ex1:.4f}")
print(f"PSI: {psi_ex1:.4f}")

ks_threshold = 0.15
psi_threshold = 0.25

if ks_ex1 > ks_threshold or psi_ex1 > psi_threshold:
    print("\nALERTA: Data Drift Detectado!")
    if ks_ex1 > ks_threshold:
        print(f"  - KS Statistic {ks_ex1:.4f} > {ks_threshold}")
    if psi_ex1 > psi_threshold:
        print(f"  - PSI {psi_ex1:.4f} > {psi_threshold}")
else:
    print("\nSem drift detectado")

print("\nO que observar:")
print(f"  - X_train media={X_train_ex1.mean():.3f}, desvio={X_train_ex1.std():.3f}")
print(f"  - X_prod media={X_prod_ex1.mean():.3f}, desvio={X_prod_ex1.std():.3f}")
print("  - Diferenca clara entre distribuicoes")

print("\nO que concluir:")
print("  Detector captura mudanca de distribuicao")


=== SOLUCAO: Exercicio 1 - Detector de Data Drift ===

KS Statistic: 0.1940
PSI: 0.3111

ALERTA: Data Drift Detectado!
  - KS Statistic 0.1940 > 0.15
  - PSI 0.3111 > 0.25

O que observar:
  - X_train media=-0.045, desvio=1.046
  - X_prod media=0.466, desvio=1.200
  - Diferenca clara entre distribuicoes

O que concluir:
  Detector captura mudanca de distribuicao


### TAREFA DO ALUNO Exercicio 2: Monitorar Performance Degradando

Simule performance caindo ao longo do tempo. Gere alerta quando cair abaixo de limiar.

**Tarefa**:
1. Gerar accuracy de 30 dias: comeca em 95%, degrada ~0.2% por dia
2. Plotar timeline com threshold (90%)
3. Identificar dia em que passou limiar
4. Disparar alerta

**Saida esperada**: Dia exato quando performance critica


In [19]:
print("\n=== SOLUCAO: Exercicio 2 - Monitorar Performance ===\n")

dias_ex2 = np.arange(0, 30)
performance_ex2 = 0.95 - 0.002 * dias_ex2 + np.random.normal(0, 0.005, len(dias_ex2))
performance_ex2 = np.clip(performance_ex2, 0, 1)

threshold_ex2 = 0.90

dia_critico = None
for dia, perf in zip(dias_ex2, performance_ex2):
    if perf < threshold_ex2:
        dia_critico = dia
        break

print(f"Performance simulada de 30 dias:")
for dia, perf in zip(dias_ex2[::3], performance_ex2[::3]):
    status = "OK" if perf >= threshold_ex2 else "ALERTA!"
    print(f"  Dia {dia:2d}: {perf:.3f} {status}")

if dia_critico is not None:
    print(f"\nALERTA: Performance caiu abaixo {threshold_ex2:.0%} no DIA {dia_critico}")
    print(f"Valor naquele dia: {performance_ex2[dia_critico]:.3f}")
else:
    print(f"\nSem alertas: performance mantida acima {threshold_ex2:.0%}")

print("\nO que observar:")
print("  - Queda consistente em performance")
print("  - Dia especifico de degradacao critica")

print("\nO que concluir:")
print("  Monitoramento detecta quando agir")


=== SOLUCAO: Exercicio 2 - Monitorar Performance ===

Performance simulada de 30 dias:
  Dia  0: 0.943 OK
  Dia  3: 0.945 OK
  Dia  6: 0.935 OK
  Dia  9: 0.939 OK
  Dia 12: 0.918 OK
  Dia 15: 0.916 OK
  Dia 18: 0.916 OK
  Dia 21: 0.908 OK
  Dia 24: 0.898 ALERTA!
  Dia 27: 0.905 OK

ALERTA: Performance caiu abaixo 90% no DIA 24
Valor naquele dia: 0.898

O que observar:
  - Queda consistente em performance
  - Dia especifico de degradacao critica

O que concluir:
  Monitoramento detecta quando agir


### TAREFA DO ALUNO Exercicio 3: Feature-wise Drift Detection

Monitore 3 features separadamente. Apenas 1 deve ter drift.

**Tarefa**:
1. Gerar 3 features de treino com distribuicao Normal(0, 1)
2. Feature 0 e 1: mesma distribuicao em producao
3. Feature 2: distribuicao diferente em producao
4. Calcular PSI/KS para cada feature
5. Identificar qual feature tem drift

**Saida esperada**: Feature 2 identificada com drift


In [20]:
print("\n=== SOLUCAO: Exercicio 3 - Feature-wise Drift ===\n")

n_feat = 3
n_samples = 300

X_train_ex3 = np.random.normal(0, 1, (n_samples, n_feat))

X_prod_ex3 = np.random.normal(0, 1, (n_samples, n_feat))
X_prod_ex3[:, 2] = np.random.normal(1.5, 0.8, n_samples)

print("Feature-wise Drift Analysis:")
print("=" * 60)

for feat in range(n_feat):
    ks_feat = ks_statistic(X_train_ex3[:, feat], X_prod_ex3[:, feat])
    psi_feat = calculate_psi_simple(X_train_ex3[:, feat], X_prod_ex3[:, feat])

    has_drift = ks_feat > 0.15 or psi_feat > 0.25
    status = "DRIFT!" if has_drift else "OK"

    print(f"\nFeature {feat}:")
    print(f"  KS:  {ks_feat:.4f}")
    print(f"  PSI: {psi_feat:.4f}")
    print(f"  Status: {status}")

print("\n" + "=" * 60)
print("Diagnostico: Feature 2 tem drift significativo")

print("\nO que observar:")
print("  - Apenas Feature 2 mostra metricas altas")
print("  - Features 0 e 1 estao normais")

print("\nO que concluir:")
print("  Drift granular ajuda encontrar causa raiz")


=== SOLUCAO: Exercicio 3 - Feature-wise Drift ===

Feature-wise Drift Analysis:

Feature 0:
  KS:  0.0867
  PSI: 0.0509
  Status: OK

Feature 1:
  KS:  0.0567
  PSI: 0.0687
  Status: OK

Feature 2:
  KS:  0.6267
  PSI: 3.4869
  Status: DRIFT!

Diagnostico: Feature 2 tem drift significativo

O que observar:
  - Apenas Feature 2 mostra metricas altas
  - Features 0 e 1 estao normais

O que concluir:
  Drift granular ajuda encontrar causa raiz


## Resumo

### Hierarquia de Conceitos

1. **Nivel 0 - Base**: Distribuicoes (cap 1), testes (cap 2)
2. **Nivel 1 - Pratica**: Deploy de modelos (cap 6.1)
3. **Nivel 2 - Monitoramento**: Drift detection (este caderno)
4. **Nivel 3 - Acoes**: Retraining, alerting, observabilidade

### Checklist de Aprendizado

- [ ] Entendo diferenca entre Data Drift e Concept Drift
- [ ] Sei calcular KS statistic e PSI
- [ ] Posso implementar detector de drift usando apenas numpy
- [ ] Entendo quando retreinar vs quando ignorar
- [ ] Sei distinguir falsos positivos (sazonalidade) de drift real
- [ ] Posso monitorar feature-wise (nao apenas global)
- [ ] Entendo importancia de logging estruturado
- [ ] Posso criar dashboard de monitoramento

### Proximos Passos

1. **Aplicar em projeto real**: Use tecnicas deste caderno em seu modelo
2. **Implementar pipeline**: Integre monitoramento com deployment
3. **Calibrar thresholds**: Ajuste PSI/KS para seu dominio
4. **Documentar baseline**: Sempre guarde dados e metricas de treino
5. **Automizar alertas**: Configure sistema de notificacoes
6. **Medir impact**: Compare sistemas com vs sem monitoramento

### Leituras Adicionais

- Evidently AI documentation (ferramentas prontas)
- whylabs.ai (monitoramento especializado)
- Neptune.ai (rastreamento de experimentos)
- Weights & Biases (observabilidade ML)

### Referencias

- Bifet & Gavaldá (2007): Learning from time-changing data
- Gama et al (2014): A survey on concept drift adaptation
- Polyzotis et al (2019): Data Lifecycle: End-to-End ML

### Palavras-chave Finais

Monitoramento nao e opcional em ML. E como manutencao preventiva: caro fazer certo, muito caro falhar.
